# **Topic Modelling**

In [11]:
# 1. (Opsional) Install dulu library yang diperlukan:
#    pip install pandas scikit-learn langdetect

import pandas as pd
from langdetect import detect
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation


In [7]:
import pandas as pd

# Asumsi: File CSV bernama 'preprocessed_data.csv' berada di direktori yang sama dengan skrip Python
df = pd.read_csv(r"D:\Internship\AIBeecara\PROJECT TEST\data_stemm_2.csv")

# Menampilkan beberapa baris pertama dari DataFrame untuk verifikasi
print(df.head())

                                             stemmed
0  use appfre version year pretti good past month...
1  pay version per year want use interact video c...
2  edit latest updat heart situat went annoy stup...
3  app present basic convers via repetit drill tr...
4  great app wish ad situat figur better week ad ...


In [12]:
# 3. Deteksi bahasa (menghasilkan 'en', 'id', atau 'unknown')
def detect_lang(text):
    try:
        return detect(text)
    except:
        return 'unknown'

df['lang'] = df['stemmed'].apply(detect_lang)

# 4. Fungsi untuk menjalankan LDA pada subset bahasa tertentu
def run_lda(texts, n_topics=5, max_df=0.9, min_df=5, n_top_words=10):
    """
    texts      : list of str (dokumen yang akan di-LDA)
    n_topics   : jumlah topik
    max_df     : abaikan kata yang muncul di > max_df proporsi dokumen
    min_df     : abaikan kata yang muncul di < min_df dokumen
    """
    # Buat Document-Term Matrix
    vect = CountVectorizer(max_df=max_df, min_df=min_df)
    dtm = vect.fit_transform(texts)
    
    # Fit LDA
    lda = LatentDirichletAllocation(
        n_components=n_topics,
        random_state=42,
        learning_method='batch'
    )
    lda.fit(dtm)
    
    # Tampilkan top words per topic
    feature_names = vect.get_feature_names_out()
    for idx, topic in enumerate(lda.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic #{idx+1}: {', '.join(top_words)}")

# 5. Jalankan untuk masing-masing bahasa
for lang_code in ['en', 'id']:
    subset = df[df['lang'] == lang_code]['stemmed'].tolist()
    if subset:
        print(f"\n=== Topics for language: {lang_code} ===")
        run_lda(
            texts=subset,
            n_topics=5,       # misalnya 5 topik per bahasa
            max_df=0.8,       # abaikan kata yang terlalu umum
            min_df=5,         # abaikan kata yang terlalu jarang
            n_top_words=10    # 10 kata teratas tiap topik
        )
    else:
        print(f"\n(no documents for language '{lang_code}')")



=== Topics for language: en ===
Topic #1: app, ad, heart, use, learn, practic, get, lesson, free, pay
Topic #2: learn, languag, app, duolingo, lesson, new, use, good, way, great
Topic #3: ad, lesson, app, get, duolingo, use, everi, pay, super, dont
Topic #4: word, learn, app, languag, like, im, use, also, speak, ive
Topic #5: learn, use, duolingo, like, make, lesson, app, get, languag, would

=== Topics for language: id ===
Topic #1: iklan, ajar, bagus, nya, salah, banget, udah, aplikasi, selesai, pas
Topic #2: nya, hati, ajar, bagus, yg, kalo, ya, aja, tolong, duolingo
Topic #3: ajar, bahasa, duolingo, aplikasi, inggris, banget, bagus, seru, main, game
Topic #4: ajar, bahasa, aplikasi, inggris, duolingo, bagus, bantu, nya, banget, mudah
Topic #5: bahasa, ajar, duolingo, aplikasi, jepang, bagus, fitur, nya, kasih, bantu
